In [7]:
"""
DSF (Differential Scanning Fluorimetry) Analysis Notebook
=======================================================

This notebook provides a comprehensive analysis pipeline for DSF data, including:
- Data loading and preprocessing
- Melting curve analysis
- Model fitting and comparison
- Statistical analysis
- Visualization of results

Required Dependencies:
--------------------
- pandas: Data manipulation and analysis
- matplotlib: Basic plotting
- numpy: Numerical computations
- scipy: Scientific computing and optimization
- plotly: Interactive visualizations
- nbformat: Notebook format handling

Installation:
------------
Create a new conda environment and install dependencies:
    conda create -n dsfanalysis
    conda activate dsfanalysis
    conda install pandas matplotlib numpy scipy plotly nbformat 
    conda install -c conda-forge python-kaleido

Then make sure you are using that environment when running this notebook. 
"""

# Standard library imports
import os
import glob

# Third-party imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
import plotly.express as px
from matplotlib.backends.backend_pdf import PdfPages
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
from scipy.signal import savgol_filter, find_peaks


# Processing Input Files

In [ ]:
#use this if you need to convert txt files to csv files

import os
import pandas as pd

# Set your parent directory here
parent_dir = "/Users/u4855540/github-repositories/dsf-thermal-shift/Annabel" 

# Define the delimiter used in the txt files (adjust if needed)
delimiter = '\t'  # Change to ',' or another delimiter if needed

for root, dirs, files in os.walk(parent_dir):
    for file in files:
        if file.endswith('.txt') and 'RawData' in file:
            txt_path = os.path.join(root, file)
            csv_path = os.path.splitext(txt_path)[0] + '.csv'

            try:
                # Read txt using pandas
                df = pd.read_csv(txt_path, delimiter=delimiter)
                df.to_csv(csv_path, index=False)
                print(f"Converted: {txt_path} -> {csv_path}")
            except Exception as e:
                print(f"Failed to convert {txt_path}: {e}")

In [21]:
# FUNCTIONS FOR LOADING DATA
import os
import glob
import re
import pandas as pd

def unify_plate_id(raw_plate_id):
    """
    Convert 'PM2A' -> 'PM2', 'PM3B' -> 'PM3', etc. 
    If it doesn't match pattern 'PM\\d+', return as-is.
    """
    match = re.match(r'^(PM\d+)', raw_plate_id)
    if match:
        return match.group(1)
    return raw_plate_id

def parse_protein_plate_from_subfolder(subfolder_name):
    """
    E.g. 'B5_PM2A' -> (protein='B5', plate_id='PM2')
    """
    parts = subfolder_name.split("_")
    if len(parts) >= 2:
        protein = parts[0]
        raw_plate_id = parts[1]
        plate_id = unify_plate_id(raw_plate_id)
    else:
        protein = subfolder_name
        plate_id = "UnknownPlate"
    return protein, plate_id

def parse_replicate_from_filename(file_name):
    """
    Look for a pattern like '_2.eds.csv' to treat as replicate #2 -> 'Rep02'.
    """
    match = re.search(r"_([0-9]+)\.eds\.csv$", file_name)
    match2 = re.search(r"_trial([0-9]+)_amb\.eds\.csv$", file_name)
    if match:
        print(f"match: {match.group(1)}")
        rep_number_str = match.group(1)
        rep_number = int(rep_number_str)
        return f"Rep{str(rep_number).zfill(2)}"
    elif match2: 
        print(f"match2: {match2.group(1)}")
        rep_number_str = match2.group(1)
        rep_number = int(rep_number_str)
        return f"Rep{str(rep_number).zfill(2)}"
    return "Rep01"

def split_raw_derivative_boltzmann_data(file_path):
    """
    Splits a multi-segmented data file into (raw_data, derivative_data, boltzmann_data).
    Assumes each segment is demarcated by lines like 'Derivative' or 'Boltzmann Temperature'.
    """
    with open(file_path, 'r', encoding='ISO-8859-1') as f:
        lines = f.readlines()

    boltzmann_start = None
    derivative_start = None
    
    for i, line in enumerate(lines):
        if "Boltzmann Temperature" in line and "Boltzmann Fluorescence" in line:
            boltzmann_start = i
            # print(f"Boltzmann start: {boltzmann_start}")
        if "Derivative" in line:
            derivative_start = i
            # print(f"Derivative start: {derivative_start}")
            
    if boltzmann_start is None:
        raise ValueError(f"No Boltzmann data found in the file: {file_path}")
    if derivative_start is None:
        raise ValueError(f"No Derivative data found in the file: {file_path}")
    
    # Read the raw data up to the derivative section
    raw_data = pd.read_csv(
        file_path, 
        sep=',', 
        encoding='ISO-8859-1', 
        nrows=derivative_start-2   # exclude header lines for derivative section
    )

    # Derivative data from derivative_start to right before boltzmann
    derivative_data = pd.read_csv(
        file_path, 
        sep=',', 
        encoding='ISO-8859-1', 
        skiprows=derivative_start, 
        nrows=boltzmann_start - derivative_start - 2
    )
    
    # Boltzmann data from boltzmann_start onward
    boltzmann_data = pd.read_csv(
        file_path, 
        sep=',', 
        encoding='ISO-8859-1', 
        skiprows=boltzmann_start - 1
    )
    
    return raw_data, derivative_data, boltzmann_data

def split_by_well_position(df, well_column="Well Position"):
    """
    Splits the given DataFrame by 'Well Position' -> { well: df_well, ... } 
    """
    well_dfs = {}
    if well_column in df.columns:
        for well in df[well_column].unique():
            subset = df[df[well_column] == well].copy()
            well_dfs[well] = subset
    return well_dfs

def load_plate_map_csvs(plate_map_folder):
    """
    Finds all CSVs in plate_map_folder, e.g. 'PM2A_PlateMap_CSV.csv',
    unifies the plate_id, and loads into plate_map_dict[plate_id] = DataFrame
    """
    import glob
    
    plate_map_dict = {}
    csv_files = glob.glob(os.path.join(plate_map_folder, "*.csv"))
    
    for csv_file in csv_files:
        base_name = os.path.basename(csv_file)
        raw_plate = base_name.split("_")[0]  # e.g. 'PM2A'
        plate_id = unify_plate_id(raw_plate)  # unify it
        df_map = pd.read_csv(csv_file)
        plate_map_dict[plate_id] = df_map
    
    return plate_map_dict

def build_well_to_ligand_map(plate_map_df, well_col="Well", ligand_col="Ligand"):
    """
    From a plate map DataFrame, build { well: ligand_name }
    """
    well_to_ligand = {}
    for idx, row in plate_map_df.iterrows():
        w = str(row[well_col]).strip()
        lig = str(row[ligand_col]).strip()
        well_to_ligand[w] = lig
    return well_to_ligand


def load_data_with_derivative_boltzmann(parent_folder, plate_map_folder):
    """
    1. Load plate map CSVs (unify plate IDs).
    2. Traverse folders: parent_folder / <protein_subfolder> / <protein>_<plate> / files...
    3. For each file, parse out (protein, plate_id, rep_id).
    4. Run split_raw_derivative_boltzmann_data() -> raw_df, derivative_df, boltzmann_df
    5. Split each by well -> store in raw_data_dfs, derivative_data_dfs, boltzmann_data_dfs
       with a 'Ligand' column added if possible.
    
    Returns:
      raw_data_dfs[protein][plate_id][rep_id][well] = DataFrame
      derivative_data_dfs[protein][plate_id][rep_id][well] = DataFrame
      boltzmann_data_dfs[protein][plate_id][rep_id][well] = DataFrame
    """
    # 1. Load plate maps & build well→ligand map
    plate_maps = load_plate_map_csvs(plate_map_folder)
    
    plate_well_ligand_map = {}
    for pid, pm_df in plate_maps.items():
        plate_well_ligand_map[pid] = build_well_to_ligand_map(pm_df, "Well", "Ligand")
    
    raw_data_dfs = {}
    derivative_data_dfs = {}
    boltzmann_data_dfs = {}
    
    # 2. Traverse subfolders
    for protein_subfolder in os.listdir(parent_folder):
        protein_subfolder_path = os.path.join(parent_folder, protein_subfolder)
        if not os.path.isdir(protein_subfolder_path):
            continue
        
        # Inside that, subfolders named like "B5_PM2A"
        for plate_subfolder in os.listdir(protein_subfolder_path):
            full_plate_subfolder_path = os.path.join(protein_subfolder_path, plate_subfolder)
            if not os.path.isdir(full_plate_subfolder_path):
                continue
            
            protein, plate_id = parse_protein_plate_from_subfolder(plate_subfolder)
            # ensure the dictionary structure
            if protein not in raw_data_dfs:
                raw_data_dfs[protein] = {}
                derivative_data_dfs[protein] = {}
                boltzmann_data_dfs[protein] = {}
            
            if plate_id not in raw_data_dfs[protein]:
                raw_data_dfs[protein][plate_id] = {}
                derivative_data_dfs[protein][plate_id] = {}
                boltzmann_data_dfs[protein][plate_id] = {}
            
            # 3. Grab all relevant data files
            data_files = glob.glob(os.path.join(full_plate_subfolder_path, "*.csv"))
            for data_file in data_files:
                file_name = os.path.basename(data_file)
                rep_id = parse_replicate_from_filename(file_name)
                print(f"working on {protein}, {plate_id}, {rep_id}")
                
                if rep_id not in raw_data_dfs[protein][plate_id]:
                
                    raw_data_dfs[protein][plate_id][rep_id] = {}
                    derivative_data_dfs[protein][plate_id][rep_id] = {}
                    boltzmann_data_dfs[protein][plate_id][rep_id] = {}
                
                # 4. Split the file into 3 DataFrames
                try:
                    raw_df, deriv_df, boltz_df = split_raw_derivative_boltzmann_data(data_file)
                except Exception as e:
                    print(f"Error parsing file {data_file}: {e}")
                    continue
                
                # 5. Split each by well & assign ligand
                
                raw_wells = split_by_well_position(raw_df, well_column="Well Position")
                deriv_wells = split_by_well_position(deriv_df, well_column="Well Position")
                boltz_wells = split_by_well_position(boltz_df, well_column="Well Position")
                # print(raw_df)
                # If we have a plate map for plate_id, get the well→ligand dict
                well_map = plate_well_ligand_map.get(plate_id, None)  # could be None if not found
                
                # Store them
                for wpos, wdf in raw_wells.items():
                    if well_map:
                        ligand = well_map.get(wpos, "UnknownLigand")
                        wdf["Ligand"] = ligand
                    raw_data_dfs[protein][plate_id][rep_id][wpos] = wdf
                
                for wpos, wdf in deriv_wells.items():
                    if well_map:
                        ligand = well_map.get(wpos, "UnknownLigand")
                        wdf["Ligand"] = ligand
                    derivative_data_dfs[protein][plate_id][rep_id][wpos] = wdf
                
                for wpos, wdf in boltz_wells.items():
                    if well_map:
                        ligand = well_map.get(wpos, "UnknownLigand")
                        wdf["Ligand"] = ligand
                    boltzmann_data_dfs[protein][plate_id][rep_id][wpos] = wdf
    
    return raw_data_dfs, derivative_data_dfs, boltzmann_data_dfs

def get_ligand_for_plate_well(raw_data_dfs, plate_id, well):
    """
    Retrieves the ligand name for a given plate and well, ignoring protein/replicate
    because the same ligand is assumed for that plate/well across all proteins/replicates.
    
    Parameters:
    - raw_data_dfs : dict
        Nested dictionary of the form:
          raw_data_dfs[protein][plate][rep_id][well] = DataFrame
        where each well DataFrame includes a 'Ligand' column.
    - plate_id : str
        e.g. "PM1"
    - well : str
        e.g. "A03"
    
    Returns:
    - A string describing the found ligand, or an error message if not found.
    """
    # Loop through all proteins in the structure
    for protein in raw_data_dfs:
        # Check if the plate_id exists for this protein
        if plate_id not in raw_data_dfs[protein]:
            continue
        
        # Loop through all replicates for this protein/plate
        for rep_id, wells_dict in raw_data_dfs[protein][plate_id].items():
            # Check if the requested well is present
            if well in wells_dict:
                df_well = wells_dict[well]
                # Ensure the DataFrame is not empty and has a 'Ligand' column
                if df_well.empty:
                    continue
                if "Ligand" not in df_well.columns:
                    return f"'Ligand' column not found in well {well} under {protein}-{plate_id} {rep_id}"
                
                # If we get here, we found the well DataFrame with a 'Ligand' column
                unique_ligands = df_well["Ligand"].unique()
                if len(unique_ligands) == 1:
                    return unique_ligands[0]
                elif len(unique_ligands) > 1:
                    return f"Multiple ligands found for {plate_id}-{well}: {unique_ligands}"
    
    # If we finish all loops without returning, it means no matching plate/well was found
    return f"No ligand data found for plate={plate_id}, well={well} in raw_data_dfs."
    

In [ ]:
# RUN DATA EXTRACTION

# Configuration for directory traversal
subdirectories = 'yes'  # Whether to look through subdirectories or not (default = 'yes')

parent_folder = "/Users/u4855540/github-repositories/dsf-thermal-shift/Annabel"        # Has top-level subfolders B5/, B6/, etc.
plate_map_folder = "/Users/u4855540/github-repositories/dsf-thermal-shift/oliver/plate_maps"    # Contains PM1_PlateMap_CSV.csv, PM2A_PlateMap_CSV.csv, etc.

raw_data_dfs, derivative_data_dfs, boltzmann_data_dfs  = load_data_with_derivative_boltzmann(parent_folder, plate_map_folder)

In [ ]:
#VISUALISE AND CHECK DATA TREE STRUCTURE
for protein in raw_data_dfs.keys():
    print(protein)
    for plate in raw_data_dfs[protein].keys():
        print(f"--| {plate}")
        for rep in raw_data_dfs[protein][plate].keys():
            print(f"----| {rep}")
    

# Examples of looking at input data

Once the above cells have been run, you will have several dataframes with your data in them. These are: 

MOST IMPORTANT: 
- `raw_data_dfs`: contains raw data traces. 

Less important: 
- `derivative_data_dfs` contains derivative data derived from RawData files if rawdata had that in it. 
- `boltzmann_data_dfs` contains boltzman data from fit from TSA analysis. 

Specific data can then be called up using: e.g. `well_results_dfs["H2"]["PM1"]["Rep01"]`


In [ ]:
raw_data_dfs["F5"]["PM1"]["Rep01"]["A03"]

In [ ]:
derivative_data_dfs["B5"]["PM1"]["Rep01"]["A03"]

In [ ]:
raw_data_dfs["B5"]["PM1"]["Rep01"]["A01"][["Temperature", "Derivative"]]


# Finding ligand name from data: 

Assuming your TSA plate layouts are correct, then you can find out ligand name using `get_ligand_for_well` function

In [ ]:
ligand = get_ligand_for_plate_well(raw_data_dfs, "PM1", "A03")
print(f"Ligand for H2 PM1 Rep01 A03: {ligand}")

# Fit models to data

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter, find_peaks
from sklearn.metrics import r2_score

# --- MODEL FUNCTIONS ---

def s1_model(x, Asym, xmid, scal, d):
    return Asym / (1 + np.exp((xmid - x) / scal)) * np.exp(d * (x - xmid))

def s1_d_model(x, Asym, xmid, scal, d, id_d, id_b):
    return s1_model(x, Asym, xmid, scal, d) + id_d * np.exp(id_b * x)

def s2_model(x, Asym, xmid, scal, d, Asym2, xmid2, scal2, d2):
    return s1_model(x, Asym, xmid, scal, d) + s1_model(x, Asym2, xmid2, scal2, d2)

def s2_d_model(x, Asym, xmid, scal, d, Asym2, xmid2, scal2, d2, id_d, id_b):
    return s2_model(x, Asym, xmid, scal, d, Asym2, xmid2, scal2, d2) + id_d * np.exp(id_b * x)

### New “modified Boltzmann” model using normalized temperature
def modified_boltzmann_model(x, A, B, C, D, xmid, E):
    """
    A two-slope-plus-sigmoid model ("modified Boltzmann"), adapted to use
    normalized temperature x in [0..1]. Interpreting 'xmid' as the
    normalized Tm, and E as the 'width' or slope factor.
    
    The form is:
      y = A*x + B + (C*x + D) / (1 + exp((xmid - x)/E))
    """
    return A*x + B + (C*x + D) / (1 + np.exp((xmid - x) / E))

# Dictionary to store model definitions and parameters.
MODEL_PARAMS = {
    "s1": {
        "model_func": s1_model,
        "p0": [1, 0.6, 0.03, -1],
        "bounds": ([0.1, 0.1, 0.01, -10], [5, 0.95, 1, 10]),
        "tm_idx": [1]
    },
    "s1_d": {
        "model_func": s1_d_model,
        "p0": [1, 0.6, 0.03, -1, 0.01, -5],
        "bounds": ([0.1, 0.1, 0.01, -10, 0.001, -20], [5, 0.95, 1, 10, 5, 0]),
        "tm_idx": [1]
    },
    "s2": {
        "model_func": s2_model,
        "p0": [1, 0.6, 0.03, -1, 0.5, 0.65, 0.03, -2],
        "bounds": ([0.01, 0.1, 0.01, -10, 0.01, 0.1, 0.01, -10],
                   [5, 0.95, 1, 10, 5, 0.95, 1, 10]),
        "tm_idx": [1, 5]
    },
    "s2_d": {
        "model_func": s2_d_model,
        "p0": [1, 0.6, 0.03, -1, 0.5, 0.65, 0.03, -2, 0.01, -5],
        "bounds": ([0.01, 0.1, 0.01, -10, 0.01, 0.1, 0.01, -10, 0.001, -20],
                   [5, 0.95, 1, 10, 5, 0.95, 1, 10, 5, 0]),
        "tm_idx": [1, 5]
    }, 
    "mB": {
        "model_func": modified_boltzmann_model,
        # Initial guess for [A, B, C, D, xmid, E].
        # xmid ~ 0.5 is near the midpoint, E ~ 0.01 is a typical slope factor, etc.
        "p0": [0, 0, 0, 0, 0.5, 0.01],
        # Bounds are just an example; tweak to fit your data scale.
        "bounds": ([-10, -10, -10, -10, 0.0, 1e-5], [10, 10, 10, 10, 1.0, 0.1]),
        # Tm is the 5th parameter in the list (index=4) 
        "tm_idx": [4]
    }
}

def fit_model_generic(model_name, x, y, custom_p0=None):
    """
    Generic model fitting function that selects the model function, initial guess, and bounds
    based on the model name from MODEL_PARAMS.
    """
    if model_name not in MODEL_PARAMS:
        raise ValueError("Unknown model name. Choose from: " + ", ".join(MODEL_PARAMS.keys()))
    
    params = MODEL_PARAMS[model_name]
    p0 = custom_p0 if custom_p0 is not None else params["p0"]
    bounds = params["bounds"]
    model_func = params["model_func"]
    
    try:
        popt, _ = curve_fit(model_func, x, y, p0=p0, bounds=bounds, maxfev=10000)
        return popt
    except Exception as e:
        print(f"[Fit failed] {model_name}: {e}")
        return None

# --- DATA PRE-PROCESSING & DERIVATIVE FUNCTIONS ---

def normalize_df(df):
    """Normalize Temperature and Fluorescence, and store original bounds as attributes."""
    df = df.copy()
    t_min, t_max = df["Temperature"].min(), df["Temperature"].max()
    f_min, f_max = df["Fluorescence"].min(), df["Fluorescence"].max()
    df["Temperature_norm"] = (df["Temperature"] - t_min) / (t_max - t_min)
    df["value_norm"] = (df["Fluorescence"] - f_min) / (f_max - f_min)
    df.attrs["T_bounds"] = (t_min, t_max)
    df.attrs["F_bounds"] = (f_min, f_max)
    return df

def find_derivative_peaks(df, window_size=11, derivative_threshold=0.0002):
    """
    Apply Savitzky-Golay filter to compute the first and second derivatives,
    and detect peaks in the first derivative.
    """
    df = df.copy()
    # Ensure an odd window size and a minimum value.
    window_size = max(5, window_size if window_size % 2 == 1 else window_size+1)
    
    if "Temperature_norm" not in df.columns or "value_norm" not in df.columns:
        df = normalize_df(df)
    
    df["sgd1"] = savgol_filter(df["value_norm"], window_size, polyorder=3, deriv=1)
    df["sgd2"] = savgol_filter(df["value_norm"], window_size, polyorder=3, deriv=2)
    peaks, _ = find_peaks(df["sgd1"], height=derivative_threshold, distance=20)
    return df, peaks

def get_deriv_peak_temp(df):
    """Return the Temperature corresponding to the highest peak in the first derivative."""
    df_with_deriv, peaks = find_derivative_peaks(df)
    if len(peaks) == 0:
        return None
    best_peak_idx = peaks[np.argmax(df_with_deriv["sgd1"].iloc[peaks].values)]
    return df_with_deriv["Temperature"].iloc[best_peak_idx]

def compute_derivative(df, window=11, polyorder=2):
    """Compute and add the first derivative to the dataframe."""
    df = df.copy().sort_values("Temperature").reset_index(drop=True)
    x = df["Temperature"].values
    y = df["value_norm"].values
    delta = np.mean(np.diff(x)) if len(x) > 1 else 1.0
    window = max(5, window if window % 2 == 1 else window+1)
    deriv = savgol_filter(y, window_length=window, polyorder=polyorder, deriv=1, delta=delta)
    df["Derivative"] = deriv
    return df


def estimate_minor_peak_temp(df, t_min, t_max, edge_margin=5, window_size=11, major_peak=None):
    """
    Estimate a candidate minor peak (for Tm2) from the second derivative.
    
    Parameters:
      df : DataFrame assumed to have columns 'Temperature', 'Temperature_norm',
           'value_norm'. (Derivatives will be computed in this function.)
      t_min, t_max : The real temperature bounds for the current window.
      edge_margin : Discard data in the first/last edge_margin °C.
      window_size : Window size for the Savitzky–Golay filter.
      major_peak : The real temperature of the major peak (Tm1); candidates within 4°C of this are ignored.
    
    Returns:
      The normalized temperature corresponding to the chosen candidate valley 
      (i.e. the best minor peak) or None if no candidate qualifies.
    """
    # Work on a copy
    df = df.copy()
    # Compute first and second derivatives on normalized fluorescence.
    df["sgd1"] = savgol_filter(df["value_norm"], window_size, polyorder=3, deriv=1)
    df["sgd2"] = savgol_filter(df["value_norm"], window_size, polyorder=3, deriv=2)
    
    # Only consider points that are not near the edges.
    valid_idxs = [i for i in range(len(df))
                  if (df["Temperature"].iloc[i] >= t_min + edge_margin and 
                      df["Temperature"].iloc[i] <= t_max - edge_margin)]
    if not valid_idxs:
        return None

    # Find valleys in sgd2 (i.e. peaks in the inverted signal)
    valleys, _ = find_peaks(-df["sgd2"])
    
    # Filter valleys: they must be in valid_idxs and have sgd1 > 0.
    candidate_idxs = []
    for i in valleys:
        if i in valid_idxs and df["sgd1"].iloc[i] > 0:
            # Convert normalized temperature to real temperature.
            T_real = df["Temperature_norm"].iloc[i] * (t_max - t_min) + t_min
            if major_peak is not None and abs(T_real - major_peak) < 4:
                continue  # Ignore candidate too close to major peak.
            candidate_idxs.append(i)
    
    if not candidate_idxs:
        return None
    
    # Choose the candidate with the most negative sgd2 (deepest trough).
    chosen_idx = min(candidate_idxs, key=lambda i: df["sgd2"].iloc[i])
    return df["Temperature_norm"].iloc[chosen_idx]


# --- MODEL FITTING & ANALYSIS FUNCTIONS ---

def explore_fit_stability(protein, plate_ID, rep, well, model_name="s1", window_sizes=[10, 15, 20, 30, 40],
                          raw_data_dfs=None, verbose=True, plot_fits=False):
    """
    For a given trace, explore how model parameters vary with different temperature windows.
    Returns a DataFrame summarizing window size, Tm values, R², and whether the fit succeeded.
    """
    try:
        df_raw = raw_data_dfs[protein][plate_ID][rep][well]
    except KeyError:
        raise ValueError(f"Missing raw data for {protein} {plate_ID} {rep} {well}")
    
    df_norm_all = normalize_df(df_raw)
    df_deriv_all = compute_derivative(df_norm_all)
    center_temp = get_deriv_peak_temp(df_deriv_all)
    if center_temp is None:
        raise ValueError("No derivative peak found for centering the window")
    
    results = []
    params = MODEL_PARAMS[model_name]
    model_func = params["model_func"]
    tm_indices = params["tm_idx"]
    
    for win in window_sizes:
        t_start = center_temp - win/2
        t_end = center_temp + win/2
        
        df_window = df_raw[(df_raw["Temperature"] >= t_start) & (df_raw["Temperature"] <= t_end)].copy()
        if len(df_window) < 10:
            if verbose:
                print(f"Window {win}°C skipped: too few data points")
            results.append({"window": win, "Tm1": None, "Tm2": None, "R2": None, "success": False})
            continue
        
        df_window_norm = normalize_df(df_window)
        x = df_window_norm["Temperature_norm"].values
        y = df_window_norm["value_norm"].values
        
        popt = fit_model_generic(model_name, x, y, custom_p0=p0_custom)
        if popt is None:
            if verbose:
                print(f"Fit failed for window {win}°C")
            results.append({"window": win, "Tm1": None, "Tm2": None, "R2": None, "success": False})
            continue
        
        y_pred = model_func(x, *popt)
        r2 = r2_score(y, y_pred)
        t_bounds = df_window_norm.attrs["T_bounds"]
        if len(tm_indices) == 1:
            Tm1 = popt[tm_indices[0]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
            Tm2 = None
        else:
            Tm1 = popt[tm_indices[0]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
            Tm2 = popt[tm_indices[1]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
        
        results.append({"window": win, "Tm1": Tm1, "Tm2": Tm2, "R2": r2, "success": True})
        
        if plot_fits:
            t_win_min, t_win_max = df_window_norm.attrs["T_bounds"]
            temp_dense = np.linspace(t_win_min, t_win_max, 300)
            temp_dense_norm = (temp_dense - t_win_min) / (t_win_max - t_win_min)
            y_fit_dense = model_func(temp_dense_norm, *popt)
            y_fit_dense_real = y_fit_dense * (df_window_norm.attrs["F_bounds"][1] - df_window_norm.attrs["F_bounds"][0]) + df_window_norm.attrs["F_bounds"][0]
            
            plt.figure(figsize=(6, 4))
            plt.scatter(df_window["Temperature"], df_window["Fluorescence"], label="Window Raw Data", color="black", s=10)
            plt.plot(temp_dense, y_fit_dense_real, label=f"{model_name} Fit (window={win}°C, R²={r2:.3f})", linestyle="--")
            plt.xlabel("Temperature (°C)")
            plt.ylabel("Fluorescence (RFU)")
            plt.title(f"Window {win}°C Fit (R²={r2:.3f})")
            plt.legend()
            plt.tight_layout()
            plt.show()
    
    return pd.DataFrame(results)



def select_best_model_for_trace(df_raw, candidate_models=None,
                                window_mode="optimized",
                                manual_range=None,
                                fixed_window_size=None,
                                window_sizes=[10, 15, 20, 30, 40],
                                verbose=True,
                                show_plot=True,
                                protein="",
                                ligand="",
                                rep="",
                                plate_ID="",
                                well=""):
    """
    For a given raw trace, try a grid of candidate models and window sizes
    based on the specified window_mode.
    ...
    """
    if candidate_models is None:
         # Include the new model in the default set
         candidate_models = list(MODEL_PARAMS.keys())  # ["s1", "s1_d", "s2", "s2_d", "mB"]
    
    results = []
    
    # For modes that require a center temperature, compute it.
    if window_mode in ["fixed", "optimized"]:
         df_norm_all = normalize_df(df_raw)
         df_deriv_all = compute_derivative(df_norm_all)
         center_temp = get_deriv_peak_temp(df_deriv_all)
         if center_temp is None:
              raise ValueError("No derivative peak found for centering the window")
    
    for model_name in candidate_models:
         if model_name not in MODEL_PARAMS:
              continue
         params = MODEL_PARAMS[model_name]
         model_func = params["model_func"]
         tm_indices = params["tm_idx"]
         
         
         # Determine window(s) based on window_mode.
         if window_mode == "full":
              t_start = df_raw["Temperature"].min()
              t_end = df_raw["Temperature"].max()
              windows = [(t_start, t_end)]
         elif window_mode == "manual":
              if manual_range is None or not isinstance(manual_range, (list, tuple)) or len(manual_range) != 2:
                    raise ValueError("For manual mode, manual_range must be provided as a tuple (t_start, t_end)")
              windows = [manual_range]
         elif window_mode == "fixed":
              if fixed_window_size is None:
                    raise ValueError("For fixed mode, fixed_window_size must be provided")
              t_start = center_temp - fixed_window_size/2
              t_end = center_temp + fixed_window_size/2
              windows = [(t_start, t_end)]
         elif window_mode == "optimized":
              windows = []
              for win in window_sizes:
                   t_start = center_temp - win/2
                   t_end = center_temp + win/2
                   windows.append((t_start, t_end))
         else:
              raise ValueError("window_mode must be one of 'full', 'manual', 'fixed', or 'optimized'")
         
         for (t_start, t_end) in windows:
              df_window = df_raw[(df_raw["Temperature"] >= t_start) & (df_raw["Temperature"] <= t_end)].copy()
              if len(df_window) < 10:
                    if verbose:
                         print(f"Model {model_name}, window {t_start}-{t_end}°C skipped: too few data points")
                    continue
              df_window_norm = normalize_df(df_window)
              x = df_window_norm["Temperature_norm"].values
              y = df_window_norm["value_norm"].values
              
              # Build custom initial parameter guess.
              p0_custom = list(MODEL_PARAMS[model_name]["p0"])
  
              # Update Tm1 using the major peak from the first derivative.
              major_peak = get_deriv_peak_temp(df_window_norm)
              if major_peak is not None:
                  t_min_window, t_max_window = df_window_norm.attrs["T_bounds"]
                  major_peak_norm = (major_peak - t_min_window) / (t_max_window - t_min_window)
                  if model_name in ["mB"]:
                    p0_custom[4] = major_peak_norm
                    print(f"[{model_name}] Tm1 initial guess (normalized): {p0_custom[4]:.4f} ({major_peak:.2f}°C)")
                  else:
                    p0_custom[1] = major_peak_norm
                    print(f"[{model_name}] Tm1 initial guess (normalized): {p0_custom[1]:.4f} ({major_peak:.2f}°C)")
                  print(f"[{model_name}] Tm1 initial guess (normalized): {p0_custom[1]:.4f} ({major_peak:.2f}°C)")
              else:
                  print(f"[{model_name}] No major peak found; using default Tm1 = {p0_custom[1]:.4f}")
  
              # For models s2 and s2_d, update Tm2 using a minor peak from the second derivative.
              if model_name in ["s2", "s2_d"]:
                  t_min_window, t_max_window = df_window_norm.attrs["T_bounds"]
                  df_deriv_window = find_derivative_peaks(df_window_norm, window_size=11)[0]
                  minor_peak_norm = estimate_minor_peak_temp(df_deriv_window, t_min_window, t_max_window,
                                                        edge_margin=5, window_size=11, major_peak=major_peak)
                  if minor_peak_norm is not None:
                      p0_custom[5] = minor_peak_norm
                      minor_peak_real = minor_peak_norm * (t_max_window - t_min_window) + t_min_window
                      print(f"[{model_name}] Tm2 initial guess (normalized): {p0_custom[5]:.4f} ({minor_peak_real:.2f}°C)")
                  else:
                      print(f"[{model_name}] No valid minor peak found in window {t_start}-{t_end}°C")
                      
              # For baseline models (_d models), update the initial baseline parameter (p0[4])
              # using the first normalized fluorescence value.
              if model_name in ["s1_d", "s2_d"]:
                  baseline_guess = y[0] if len(y) > 0 else 0.01
                  p0_custom[4] = max(baseline_guess, 0.001)
                  print(f"[{model_name}] Baseline (id_d) initial guess: {p0_custom[4]:.4f}")
  
              popt = fit_model_generic(model_name, x, y)
              if popt is None:
                    if verbose:
                         print(f"Model {model_name}, window {t_start}-{t_end}°C: fit failed")
                    continue
              
              y_pred = model_func(x, *popt)
              r2 = r2_score(y, y_pred)
              t_bounds = df_window_norm.attrs["T_bounds"]
              if len(tm_indices) == 1:
                    Tm1 = popt[tm_indices[0]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
                    Tm2 = None
              else:
                    Tm1 = popt[tm_indices[0]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
                    Tm2 = popt[tm_indices[1]] * (t_bounds[1] - t_bounds[0]) + t_bounds[0]
              
              results.append({"model": model_name, "t_start": t_start, "t_end": t_end,
                              "R2": r2, "Tm1": Tm1, "Tm2": Tm2, "popt": popt})
              if verbose:
                    extra = f", Tm2={Tm2:.2f}" if Tm2 is not None else ""
                    print(f"Model {model_name}, window {t_start}-{t_end}°C: R²={r2:.3f}, Tm1={Tm1:.2f}{extra}")
    
    results_df = pd.DataFrame(results)
    if results_df.empty:
         best_fit = None
    else:
         best_fit = results_df.loc[results_df["R2"].idxmax()]
    
    # If requested, plot the best fit using the stored popt.
    if show_plot and best_fit is not None:
         best_model = best_fit["model"]
         best_t_start = best_fit["t_start"]
         best_t_end = best_fit["t_end"]
         best_popt = best_fit["popt"]
         best_fit_dict = {best_model: best_popt}
         
         # Normalize the full raw data (to show in background)
         full_df_norm = normalize_df(df_raw)
         print("\nPlotting best model fit on the best window (with full raw data shown):")
         plot_fits(
             normalize_df(df_raw[(df_raw["Temperature"] >= best_t_start) & (df_raw["Temperature"] <= best_t_end)]),
             best_fit_dict, full_data_df=full_df_norm,
             protein=protein, ligand=ligand, rep=rep, plate_ID=plate_ID, well=well
         )
    
    return best_fit, results_df


def plot_fits(fit_df, fits_dict, full_data_df=None, protein="", ligand="", rep="", plate_ID="", well=""):
    """
    Plot raw data and fitted curves.
    
    Parameters:
      fit_df: The normalized DataFrame for the fit window.
      fits_dict: Dictionary of {model_name: fit_parameters} from the generic fitter.
      full_data_df: A normalized DataFrame for the full raw data range (for background scatter).
      protein, ligand, rep, plate_ID, well: Strings used to build the plot title.
    
    The legend for each fitted model includes its Tm value(s) and R² score.
    For s2 and s2_d models, Tm2 is also displayed and a vertical dotted line is drawn at Tm2.
    The plot title is set to "{protein} - {ligand} {rep}" with subtitle "{plate_ID} {well}".
    """
    # Use full_data_df if provided; otherwise, use fit_df.
    raw_df = full_data_df if full_data_df is not None else fit_df

    t_min_raw, t_max_raw = raw_df.attrs["T_bounds"]
    f_min_raw, f_max_raw = raw_df.attrs["F_bounds"]
    
    # For the fitted model curve, use the fit window bounds.
    t_min_fit, t_max_fit = fit_df.attrs["T_bounds"]
    f_min_fit, f_max_fit = fit_df.attrs["F_bounds"]
    
    # Create a dense grid over the fit window.
    temp_dense_fit = np.linspace(t_min_fit, t_max_fit, 300)
    temp_dense_fit_norm = (temp_dense_fit - t_min_fit) / (t_max_fit - t_min_fit)
    
    fig, (ax_main, ax_deriv) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                            gridspec_kw={'height_ratios': [3, 1]})
    # Set the main title and subtitle.
    main_title = f"{protein}, {ligand} ({plate_ID} {well}) - {rep}  "
    fig.suptitle(main_title, fontsize=12, y=0.98)
    
    # Plot full raw data (scatter with alpha=0.8).
    ax_main.scatter(raw_df["Temperature"], raw_df["Fluorescence"], label="Raw data",
                    color="black", s=10, alpha=0.8, zorder=2)
    
    # Plot each fitted model.
    for model_name, popt in fits_dict.items():
        if popt is None:
            continue
        params = MODEL_PARAMS[model_name]
        model_func = params["model_func"]
        y_model_fit = model_func(temp_dense_fit_norm, *popt)
        y_model_fit_real = y_model_fit * (f_max_fit - f_min_fit) + f_min_fit
        x_fit = fit_df["Temperature_norm"].values
        y_fit = fit_df["value_norm"].values
        y_pred_fit = model_func(x_fit, *popt)
        r2_val = r2_score(y_fit, y_pred_fit)
        
        if model_name in ["s1", "s1_d"]:
            # single Tm at popt[1]
            tm1 = popt[1] * (t_max_fit - t_min_fit) + t_min_fit
            label = f"{model_name} (Tm = {tm1:.1f}°C; R² = {r2_val:.3f})"
            # draw vertical line at tm1...
        elif model_name in ["mB"]:
            # single Tm at popt[4]
            tm1 = popt[4] * (t_max_fit - t_min_fit) + t_min_fit
            label = f"{model_name} (Tm = {tm1:.1f}°C; R² = {r2_val:.3f})"
            # draw vertical line at tm1...
        else:
            # for s2, s2_d => 2 Tms at popt[1], popt[5]
            tm1 = popt[1] * (t_max_fit - t_min_fit) + t_min_fit
            tm2 = popt[5] * (t_max_fit - t_min_fit) + t_min_fit
            label = f"{model_name} (Tm1 = {tm1:.1f}°C, Tm2 = {tm2:.1f}°C; R² = {r2_val:.3f})"
            ax_main.axvline(x=tm2, linestyle=":", color="blue", zorder=9)
        
        # Plot model curve with high zorder.
        ax_main.plot(temp_dense_fit, y_model_fit_real, label=label, linestyle="--", zorder=10)
        # Draw vertical dashed line at Tm1.
        ax_main.axvline(x=tm1, linestyle="--", color="gray", zorder=9)
    
    ax_main.set_ylabel("Fluorescence (RFU)")
    ax_main.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    ax_main.legend(fontsize=9)
    
    # For derivative plotting, use the full raw data.
    df_deriv = find_derivative_peaks(raw_df)[0]
    # Convert normalized derivative to unnormalized.
    norm_factor = (f_max_raw - f_min_raw)
    df_deriv["sgd1"] = df_deriv["sgd1"] * norm_factor
    
    ax_deriv.plot(df_deriv["Temperature"], df_deriv["sgd1"], color="black", label="1st Derivative", zorder=3)
    # Filter peaks with find_peaks.
    peak_idxs = find_peaks(df_deriv["sgd1"])[0]
    if len(peak_idxs) > 0:
        peak_vals = df_deriv["sgd1"].iloc[peak_idxs].values
        global_idx = peak_idxs[np.argmax(peak_vals)]
        for peak in peak_idxs:
            temp = df_deriv["Temperature"].iloc[peak]
            dval = df_deriv["sgd1"].iloc[peak]
            if dval < 0.0002 * norm_factor:
                continue
            if peak == global_idx:
                ax_deriv.plot(temp, dval, 'o', color='red', markersize=10,
                              markeredgecolor='black', zorder=4)
                ax_deriv.text(temp, dval + (0.01 * norm_factor), f"{temp:.1f}", fontsize=9,
                              ha='center', color='black', fontweight='bold', zorder=5)
            else:
                ax_deriv.plot(temp, dval, 'o', color='red', markersize=6,
                              markeredgecolor='black', zorder=4)
                ax_deriv.text(temp, dval + (0.01 * norm_factor), f"{temp:.1f}", fontsize=8, ha='center',
                              color='black', zorder=5)
    
    ax_deriv.set_xlabel("Temperature (°C)")
    ax_deriv.set_ylabel("dF/dT (RFU/°C)")
    y_vals = df_deriv["sgd1"].values
    y_min_val, y_max_val = np.min(y_vals), np.max(y_vals)
    y_range = y_max_val - y_min_val
    ax_deriv.set_ylim(y_min_val - 0.1 * y_range, y_max_val + 0.3 * y_range)
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.show()

## EXAMPLE USAGE

In [ ]:
protein="F5"
plate="PM1"
rep="Rep01"
well="A10"

ligand=get_ligand_for_plate_well(raw_data_dfs,plate,well)
df_raw_example = raw_data_dfs[protein][plate][rep][well]

best_fit, summary_df = select_best_model_for_trace(
    df_raw_example,
    candidate_models=["mB", "s1"],
    window_mode="optimized",
    window_sizes=[20, 25, 30,40, 50, 60, 90],
    verbose=True,
    show_plot=True,  # This will trigger plotting using the full raw data
    protein=protein,
    plate_ID=plate, 
    rep=rep,
    well=well, 
    ligand=ligand
)
print("Best fit result:")
print(best_fit)
print("\nSummary of all fits:")
print(summary_df)

# PLOT ALL REPS TOGETHER

In [27]:
def plot_reps_and_neg_control(protein, plate, treatment_well, neg_control_well="A01",
                              raw_data_dfs=None, candidate_models=None,
                              window_mode="optimized", window_sizes=[10, 15, 20, 30, 40],
                              fixed_window_size=20, verbose=True,
                              include_tmchange_subplot=True,   # NEW: include top Tm-change subplot
                              include_norm_subplot=False,
                              save_png=False, png_filename=None, dTm_threshold=2):
    """
    Plot replicates for a given protein, plate, and treatment well with corresponding negative control.
    
    For each replicate, this function:
      - Retrieves raw data for treatment and negative control.
      - Uses select_best_model_for_trace to obtain model-fit parameters (Tm, R²) for both traces.
      - Plots:
         • A main (raw) data plot (ax_main) with treatment (solid line) and negative control (dashed line)
           along with their fitted curves (dotted lines) and Tm markers (dots with black edge;
           negative control markers are semi-transparent).
         • A derivative plot (ax_deriv) showing computed dF/dT curves for both traces (treatment: solid; control: dashed)
           with global peak markers and Tm (and ΔTm in the legend).
      - Optionally, if include_norm_subplot is True, adds two additional subplots (for normalized raw and normalized derivative data).
      - Additionally, if include_tmchange_subplot is True (default), adds a top subplot (ax_tmchange) that, for each replicate,
        plots horizontal lines comparing the experimental and negative-control Tm for both the model-fit and derivative methods.
        The corresponding ΔTm values are annotated; if the experimental ΔTm is positive and ≥ dTm_threshold, the label is bolded.
    
    The overall figure title is formatted as:
       {protein}, {ligand} ({plate}, {treatment_well})
    (with ligand taken from the first valid replicate).
    
    Additional Parameters:
      save_png: bool, optional – if True, saves the figure as a PNG.
      png_filename: str, optional – filename for saving; if None, a default name is used.
      dTm_threshold: float, optional – threshold in °C above which a positive ΔTm is bolded.
    
    Returns:
      None. The function displays (and optionally saves) the figure.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.signal import find_peaks

    # Use a color-blind friendly palette.
    treatment_colors = ["#0072B2", "#E69F00", "#009E73", "#F0E442", "#56B4E9", "#D55E00", "#CC79A7"]
    if candidate_models is None:
        candidate_models = list(MODEL_PARAMS.keys())
    
    # Get replicates for the given protein and plate.
    reps = list(raw_data_dfs[protein][plate].keys())
    if verbose:
        print(f"Found replicates for {protein} {plate}: {reps}")
    
    first_ligand = None

    # Decide on subplot layout.
    if include_tmchange_subplot and include_norm_subplot:
        nrows = 5
        height_ratios = [1, 3, 3, 1, 1]
        fig, axs = plt.subplots(nrows, 1, figsize=(10, 14), sharex=True,
                                gridspec_kw={'height_ratios': height_ratios})
        ax_tmchange, ax_main, ax_norm, ax_deriv, ax_deriv_norm = axs
    elif include_tmchange_subplot and not include_norm_subplot:
        nrows = 3
        height_ratios = [1, 3, 1]
        fig, axs = plt.subplots(nrows, 1, figsize=(10, 10), sharex=True,
                                gridspec_kw={'height_ratios': height_ratios})
        ax_tmchange, ax_main, ax_deriv = axs
    elif not include_tmchange_subplot and include_norm_subplot:
        nrows = 4
        height_ratios = [3, 3, 1, 1]
        fig, axs = plt.subplots(nrows, 1, figsize=(10, 12), sharex=True,
                                gridspec_kw={'height_ratios': height_ratios})
        ax_main, ax_norm, ax_deriv, ax_deriv_norm = axs
    else:
        nrows = 2
        height_ratios = [3, 1]
        fig, axs = plt.subplots(nrows, 1, figsize=(10, 8), sharex=True,
                                gridspec_kw={'height_ratios': height_ratios})
        ax_main, ax_deriv = axs

    # If using the Tm-change subplot, prepare to assign a unique y-coordinate per replicate.
    if include_tmchange_subplot:
        tmchange_ax = ax_tmchange
        tmchange_ax.set_ylabel("Replicate")
        count = 0  # y-coordinate counter

    # Process each replicate.
    for i, rep in enumerate(reps):
        rep_data = raw_data_dfs[protein][plate][rep]
        if treatment_well not in rep_data or neg_control_well not in rep_data:
            if verbose:
                print(f"{rep}: Missing required wells. Skipping.")
            continue

        df_treat = rep_data[treatment_well]
        df_neg = rep_data[neg_control_well]

        ligand_treat = get_ligand_for_plate_well(raw_data_dfs, plate, treatment_well)
        if first_ligand is None:
            first_ligand = ligand_treat
        ligand_neg = get_ligand_for_plate_well(raw_data_dfs, plate, neg_control_well)

        # Get model fits.
        best_fit_t, _ = select_best_model_for_trace(df_treat, candidate_models=candidate_models,
                                                    window_mode=window_mode, window_sizes=window_sizes,
                                                    show_plot=False, protein=protein, ligand=ligand_treat,
                                                    rep=rep, plate_ID=plate, well=treatment_well,
                                                    fixed_window_size=fixed_window_size)
        best_fit_neg, _ = select_best_model_for_trace(df_neg, candidate_models=candidate_models,
                                                      window_mode=window_mode, window_sizes=window_sizes,
                                                      show_plot=False, protein=protein, ligand=ligand_neg,
                                                      rep=rep, plate_ID=plate, well=neg_control_well,
                                                      fixed_window_size=fixed_window_size)
        rep_color = treatment_colors[i % len(treatment_colors)]

        # --- Raw Data Plot (ax_main) ---
        ax_main.plot(df_treat["Temperature"], df_treat["Fluorescence"],
                     color=rep_color, alpha=0.7, linestyle="-", linewidth=3,
                     label=f"{ligand_treat}")
        if best_fit_t is not None:
            t_start_fit = best_fit_t["t_start"]
            t_end_fit = best_fit_t["t_end"]
            df_treat_window = df_treat[(df_treat["Temperature"] >= t_start_fit) &
                                       (df_treat["Temperature"] <= t_end_fit)].copy()
            df_treat_window_norm = normalize_df(df_treat_window)
            t_min_fit, t_max_fit = df_treat_window_norm.attrs["T_bounds"]
            f_min_fit, f_max_fit = df_treat_window_norm.attrs["F_bounds"]

            popt_t = best_fit_t["popt"]
            model_func_t = MODEL_PARAMS[best_fit_t["model"]]["model_func"]
            temp_dense = np.linspace(t_min_fit, t_max_fit, 300)
            temp_dense_norm = (temp_dense - t_min_fit) / (t_max_fit - t_min_fit)
            y_fit_norm = model_func_t(temp_dense_norm, *popt_t)
            y_fit = y_fit_norm * (f_max_fit - f_min_fit) + f_min_fit

            # Calculate model-based ΔTm.
            if best_fit_neg is not None:
                dTm_model = best_fit_t["Tm1"] - best_fit_neg["Tm1"]
            else:
                dTm_model = None
            label_model = f"{ligand_treat} Fit (Tm={best_fit_t['Tm1']:.1f}°C, R²={best_fit_t['R2']:.3f}"
            if dTm_model is not None:
                label_model += f", dTm={dTm_model:.1f}°C)"
            else:
                label_model += ")"
            if dTm_model is not None and dTm_model >= dTm_threshold:
                label_model = r"$\mathbf{" + label_model + "}$"
            ax_main.plot(temp_dense, y_fit, color="black", linestyle=":", label=label_model)

            t_norm_Tm = (best_fit_t["Tm1"] - t_min_fit) / (t_max_fit - t_min_fit)
            y_fit_Tm_raw = model_func_t(t_norm_Tm, *popt_t) * (f_max_fit - f_min_fit) + f_min_fit
            ax_main.plot(best_fit_t["Tm1"], y_fit_Tm_raw, 'o', color=rep_color, markersize=8,
                         markeredgecolor='black')
            ax_main.text(best_fit_t["Tm1"], y_fit_Tm_raw + 0.05*(f_max_fit - f_min_fit),
                         f"{best_fit_t['Tm1']:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
        ax_main.plot(df_neg["Temperature"], df_neg["Fluorescence"],
                     color=rep_color, alpha=0.7, linestyle="--", linewidth=3, label=f"{ligand_neg}")
        if best_fit_neg is not None:
            t_start_fit_neg = best_fit_neg["t_start"]
            t_end_fit_neg = best_fit_neg["t_end"]
            df_neg_window = df_neg[(df_neg["Temperature"] >= t_start_fit_neg) &
                                   (df_neg["Temperature"] <= t_end_fit_neg)].copy()
            df_neg_window_norm = normalize_df(df_neg_window)
            t_min_fit_neg, t_max_fit_neg = df_neg_window_norm.attrs["T_bounds"]
            f_min_fit_neg, f_max_fit_neg = df_neg_window_norm.attrs["F_bounds"]

            popt_neg = best_fit_neg["popt"]
            model_func_neg = MODEL_PARAMS[best_fit_neg["model"]]["model_func"]
            temp_dense_neg = np.linspace(t_min_fit_neg, t_max_fit_neg, 300)
            temp_dense_neg_norm = (temp_dense_neg - t_min_fit_neg) / (t_max_fit_neg - t_min_fit_neg)
            y_fit_neg_norm = model_func_neg(temp_dense_neg_norm, *popt_neg)
            y_fit_neg = y_fit_neg_norm * (f_max_fit_neg - f_min_fit_neg) + f_min_fit_neg

            ax_main.plot(temp_dense_neg, y_fit_neg, color="black", linestyle=":",
                         label=f"{ligand_neg} Fit (Tm={best_fit_neg['Tm1']:.1f}°C, R²={best_fit_neg['R2']:.3f})")

            t_norm_Tm_neg = (best_fit_neg["Tm1"] - t_min_fit_neg) / (t_max_fit_neg - t_min_fit_neg)
            y_fit_Tm_neg_raw = model_func_neg(t_norm_Tm_neg, *popt_neg) * (f_max_fit_neg - f_min_fit_neg) + f_min_fit_neg
            ax_main.plot(best_fit_neg["Tm1"], y_fit_Tm_neg_raw, 'o', color=rep_color, markersize=8,
                         markeredgecolor='black', alpha=0.5)
            ax_main.text(best_fit_neg["Tm1"], y_fit_Tm_neg_raw + 0.05*(f_max_fit_neg - f_min_fit_neg),
                         f"{best_fit_neg['Tm1']:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
        
        # --- Normalized Raw Data Plot (ax_norm) ---
        if include_norm_subplot:
            df_treat_norm_all = normalize_df(df_treat)
            f_min_all, f_max_all = df_treat_norm_all.attrs["F_bounds"]
            ax_norm.plot(df_treat_norm_all["Temperature"], df_treat_norm_all["value_norm"],
                         color=rep_color, alpha=0.7, linestyle="-", linewidth=3,
                         label=f"{ligand_treat}")
            if best_fit_t is not None:
                y_fit_norm_full = (y_fit_norm * (f_max_fit - f_min_fit) + f_min_fit - f_min_all) / (f_max_all - f_min_all)
                label_model_norm = f"{ligand_treat} Fit (Tm={best_fit_t['Tm1']:.1f}°C, R²={best_fit_t['R2']:.3f}"
                if dTm_model is not None:
                    label_model_norm += f", dTm={dTm_model:.1f}°C)"
                else:
                    label_model_norm += ")"
                if dTm_model is not None and dTm_model >= dTm_threshold:
                    label_model_norm = r"$\mathbf{" + label_model_norm + "}$"
                ax_norm.plot(temp_dense, y_fit_norm_full, color="black", linestyle=":", label=label_model_norm)
                y_fit_Tm_norm_full = (y_fit_Tm_raw - f_min_all) / (f_max_all - f_min_all)
                ax_norm.plot(best_fit_t["Tm1"], y_fit_Tm_norm_full, 'o', color=rep_color, markersize=8,
                             markeredgecolor='black')
                ax_norm.text(best_fit_t["Tm1"], y_fit_Tm_norm_full + 0.05, f"{best_fit_t['Tm1']:.1f}",
                             fontsize=9, ha='center', color='black', fontweight='bold')
            df_neg_norm_all = normalize_df(df_neg)
            f_min_all_neg, f_max_all_neg = df_neg_norm_all.attrs["F_bounds"]
            ax_norm.plot(df_neg_norm_all["Temperature"], df_neg_norm_all["value_norm"],
                         color=rep_color, alpha=0.7, linestyle="--", linewidth=3,
                         label=f"{ligand_neg}")
            if best_fit_neg is not None:
                y_fit_neg_norm_full = (y_fit_neg_norm * (f_max_fit_neg - f_min_fit_neg) + f_min_fit_neg - f_min_all_neg) / (f_max_all_neg - f_min_all_neg)
                ax_norm.plot(temp_dense_neg, y_fit_neg_norm_full, color="black", linestyle=":",
                             label=f"{ligand_neg} Fit (Tm={best_fit_neg['Tm1']:.1f}°C, R²={best_fit_neg['R2']:.3f})")
                t_norm_Tm_neg = (best_fit_neg["Tm1"] - t_min_fit_neg) / (t_max_fit_neg - t_min_fit_neg)
                y_fit_Tm_neg_norm_full = (model_func_neg(t_norm_Tm_neg, *popt_neg)*(f_max_fit_neg - f_min_fit_neg) + f_min_fit_neg - f_min_all_neg) / (f_max_all_neg - f_min_all_neg)
                ax_norm.plot(best_fit_neg["Tm1"], y_fit_Tm_neg_norm_full, 'o', color=rep_color, markersize=8,
                             markeredgecolor='black', alpha=0.5)
                ax_norm.text(best_fit_neg["Tm1"], y_fit_Tm_neg_norm_full + 0.05, f"{best_fit_neg['Tm1']:.1f}",
                             fontsize=9, ha='center', color='black', fontweight='bold')
        
        # --- Derivative Plot (ax_deriv) ---
        df_treat_norm_full = normalize_df(df_treat)
        df_treat_deriv = compute_derivative(df_treat_norm_full)
        norm_factor = (df_treat_norm_full.attrs["F_bounds"][1] - df_treat_norm_full.attrs["F_bounds"][0])
        df_treat_deriv["Derivative"] *= norm_factor
        df_neg_norm_full = normalize_df(df_neg)
        df_neg_deriv = compute_derivative(df_neg_norm_full)
        norm_factor_neg = (df_neg_norm_full.attrs["F_bounds"][1] - df_neg_norm_full.attrs["F_bounds"][0])
        df_neg_deriv["Derivative"] *= norm_factor_neg
        
        peaks_t = find_peaks(df_treat_deriv["Derivative"])[0]
        temp_global_t = df_treat_deriv["Temperature"].iloc[peaks_t[np.argmax(df_treat_deriv["Derivative"].iloc[peaks_t])]] if len(peaks_t) > 0 else None
        peaks_neg = find_peaks(df_neg_deriv["Derivative"])[0]
        temp_global_neg = df_neg_deriv["Temperature"].iloc[peaks_neg[np.argmax(df_neg_deriv["Derivative"].iloc[peaks_neg])]] if len(peaks_neg) > 0 else None
        dTm_deriv = (temp_global_t - temp_global_neg) if (temp_global_t is not None and temp_global_neg is not None) else None
        label_t_deriv = f"{ligand_treat}"
        if temp_global_t is not None:
            label_t_deriv += f" (Tm={temp_global_t:.1f}°C"
            if dTm_deriv is not None:
                label_t_deriv += f", dTm={dTm_deriv:.1f}°C)"
            else:
                label_t_deriv += ")"
        if dTm_deriv is not None and dTm_deriv >= dTm_threshold:
            label_t_deriv = r"$\mathbf{" + label_t_deriv + "}$"
        ax_deriv.plot(df_treat_deriv["Temperature"], df_treat_deriv["Derivative"],
                      color=rep_color, linestyle="-", label=label_t_deriv)
        if len(peaks_t) > 0:
            t_pt = df_treat_deriv["Temperature"].iloc[peaks_t[np.argmax(df_treat_deriv["Derivative"].iloc[peaks_t])]]
            d_pt = df_treat_deriv["Derivative"].iloc[peaks_t[np.argmax(df_treat_deriv["Derivative"].iloc[peaks_t])]]
            ax_deriv.plot(t_pt, d_pt, 'o', color=rep_color, markersize=8, markeredgecolor='black')
            ax_deriv.text(t_pt, d_pt + 0.01*norm_factor, f"{t_pt:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
        
        label_neg_deriv = f"{ligand_neg}"
        ax_deriv.plot(df_neg_deriv["Temperature"], df_neg_deriv["Derivative"],
                      color=rep_color, linestyle=":", label=label_neg_deriv)
        if len(peaks_neg) > 0:
            t_pt_neg = df_neg_deriv["Temperature"].iloc[peaks_neg[np.argmax(df_neg_deriv["Derivative"].iloc[peaks_neg])]]
            d_pt_neg = df_neg_deriv["Derivative"].iloc[peaks_neg[np.argmax(df_neg_deriv["Derivative"].iloc[peaks_neg])]]
            ax_deriv.plot(t_pt_neg, d_pt_neg, 'o', color=rep_color, markersize=8, markeredgecolor='black')
            ax_deriv.text(t_pt_neg, d_pt_neg + 0.01*norm_factor_neg, f"{t_pt_neg:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
        cur_ylim = ax_deriv.get_ylim()
        ax_deriv.set_ylim(cur_ylim[0], cur_ylim[1] + 0.1*(cur_ylim[1]-cur_ylim[0]))
        
        # --- Normalized Derivative Plot (ax_deriv_norm) ---
        if include_norm_subplot:
            df_treat_deriv_norm = compute_derivative(df_treat_norm_full)
            label_t_deriv_norm = f"{ligand_treat}"
            if temp_global_t is not None:
                label_t_deriv_norm += f" (Tm={temp_global_t:.1f}°C"
                if dTm_deriv is not None:
                    label_t_deriv_norm += f", dTm={dTm_deriv:.1f}°C)"
                else:
                    label_t_deriv_norm += ")"
            if dTm_deriv is not None and dTm_deriv >= dTm_threshold:
                label_t_deriv_norm = r"$\mathbf{" + label_t_deriv_norm + "}$"
            ax_deriv_norm.plot(df_treat_deriv_norm["Temperature"], df_treat_deriv_norm["Derivative"],
                               color=rep_color, linestyle="-", label=label_t_deriv_norm)
            peaks_t_norm = find_peaks(df_treat_deriv_norm["Derivative"])[0]
            if len(peaks_t_norm) > 0:
                t_pt_norm = df_treat_deriv_norm["Temperature"].iloc[peaks_t_norm[np.argmax(df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm])]]
                d_pt_norm = df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm[np.argmax(df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm])]]
                ax_deriv_norm.plot(t_pt_norm, d_pt_norm, 'o', color=rep_color, markersize=8, markeredgecolor='black')
                ax_deriv_norm.text(t_pt_norm, d_pt_norm + 0.01, f"{t_pt_norm:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
            
            df_neg_deriv_norm = compute_derivative(df_neg_norm_full)
            label_neg_deriv_norm = f"{ligand_neg}"
            ax_deriv_norm.plot(df_neg_deriv_norm["Temperature"], df_neg_deriv_norm["Derivative"],
                               color=rep_color, linestyle=":", label=label_neg_deriv_norm)
            peaks_neg_norm = find_peaks(df_neg_deriv_norm["Derivative"])[0]
            if len(peaks_neg_norm) > 0:
                t_pt_neg_norm = df_neg_deriv_norm["Temperature"].iloc[peaks_neg_norm[np.argmax(df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm])]]
                d_pt_neg_norm = df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm[np.argmax(df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm])]]
                ax_deriv_norm.plot(t_pt_neg_norm, d_pt_neg_norm, 'o', color=rep_color, markersize=8,
                                   markeredgecolor='black', alpha=0.5)
                ax_deriv_norm.text(t_pt_neg_norm, d_pt_neg_norm + 0.01, f"{t_pt_neg_norm:.1f}",
                                   fontsize=9, ha='center', color='black', fontweight='bold')
            ax_deriv_norm.set_xlabel("Temperature (°C)")
       

        # --- Tm-Change Comparison Subplot (ax_tmchange) ---
        if include_tmchange_subplot:
            y_coord = count  # assign a unique vertical position
            # Model-based Tm comparison.
            if best_fit_t is not None and best_fit_neg is not None:
                exp_tm_model = best_fit_t["Tm1"]
                ctrl_tm_model = best_fit_neg["Tm1"]
                y_model = y_coord - 0.2
                tmchange_ax.annotate("", xy=(exp_tm_model, y_model), xytext=(ctrl_tm_model, y_model),
                                     arrowprops=dict(arrowstyle="->", color="black"))
                # tmchange_ax.plot([ctrl_tm_model, exp_tm_model], [y_model, y_model], linestyle='dotted', color='black')
                tmchange_ax.scatter(ctrl_tm_model, y_model, color='black')
                tmchange_ax.scatter(exp_tm_model, y_model, color=rep_color, edgecolor='black')
                delta_model = exp_tm_model - ctrl_tm_model
                label_model_change = f"{ligand_treat} Model ΔTm={delta_model:.1f}"
                if delta_model >= dTm_threshold:
                    label_model_change = r"$\mathbf{" + label_model_change + "}$"
                tmchange_ax.text(max(exp_tm_model, ctrl_tm_model)+0.5, y_model, label_model_change, va='center', fontsize=8)
            # Derivative-based Tm comparison.
            if temp_global_t is not None and temp_global_neg is not None:
                y_deriv = y_coord + 0.2
                tmchange_ax.annotate("", xy=(temp_global_t, y_deriv), xytext=(temp_global_neg, y_deriv),
                                     arrowprops=dict(arrowstyle="->", color="black"))
                # tmchange_ax.plot([temp_global_neg, temp_global_t], [y_deriv, y_deriv], linestyle='dotted', color='black')
                tmchange_ax.scatter(temp_global_neg, y_deriv, color='black')
                tmchange_ax.scatter(temp_global_t, y_deriv, color=rep_color, edgecolor='black')
                delta_deriv = temp_global_t - temp_global_neg
                label_deriv_change = f"{ligand_treat} Deriv ΔTm={delta_deriv:.1f}"
                if delta_deriv >= dTm_threshold:
                    label_deriv_change = r"$\mathbf{" + label_deriv_change + "}$"
                tmchange_ax.text(max(temp_global_t, temp_global_neg)+0.5, y_deriv, label_deriv_change, va='center', fontsize=8)
            count += 1

    # Set y-ticks for Tm-change subplot if used.
    if include_tmchange_subplot:
        tmchange_ax.set_yticks(range(count))
        tmchange_ax.set_yticklabels([f"Rep {j+1}" for j in range(count)])
    
    # Set axis labels and legends.
    ax_main.set_xlabel("")
    ax_main.set_ylabel("Fluorescence (RFU)")
    ax_main.legend(fontsize=8, loc="upper left")
    
    if include_norm_subplot:
        ax_norm.set_xlabel("")
        ax_norm.set_ylabel("Normalized Fluorescence")
        ax_norm.legend(fontsize=8, loc="upper left")
        ax_deriv.set_xlabel("")
        ax_deriv.set_ylabel("dF/dT (RFU/°C)")
        ax_deriv.legend(fontsize=8, loc="upper left")
        ax_deriv_norm.set_ylabel("Normalized dF/dT")
        ax_deriv_norm.legend(fontsize=8, loc="upper left")
    else:
        ax_deriv.set_xlabel("Temperature (°C)")
        ax_deriv.set_ylabel("dF/dT (RFU/°C)")
        ax_deriv.legend(fontsize=8, loc="upper left")
    
    if first_ligand is None:
        first_ligand = "Unknown Ligand"
    fig.suptitle(f"{protein}, {first_ligand} ({plate}, {treatment_well})", fontsize=14, y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    if save_png:
        if png_filename is None:
            png_filename = f"./output_pngs/{protein}_{plate}_{treatment_well}_allreps.png"
        plt.savefig(png_filename, dpi=300)
        if verbose:
            print(f"Figure saved as {png_filename}")
    plt.show()

### SINGLE CONDITION:

In [ ]:
protein = "B5"
plate = "PM1"
treatment_well = "B08"# treatment well input
# (Assuming get_ligand_for_well and raw_data_dfs are already defined in your environment)
plot_reps_and_neg_control(protein, plate, treatment_well, neg_control_well="A01", raw_data_dfs=raw_data_dfs, candidate_models=["mB"], window_mode="optimized", window_sizes=[30, 40, 50, 60, 70], verbose=True, include_norm_subplot=True, save_png=True)

# ANALYSE AND PLOT ALL DATA

In [28]:
def plot_reps_and_neg_control_sep(
    protein, plate, treatment_well, neg_control_well="A01",
    raw_data_dfs=None, candidate_models=None,
    window_mode="optimized", window_sizes=[20, 30, 40, 50, 60, 70],
    fixed_window_size=20, verbose=True,
    include_tmchange_subplot=True,   # include top Tm-change subplot
    include_norm_subplot=False,
    save_png=False, png_filename=None, dTm_threshold=2,
    save_metadata_csv=False, metadata_csv_filename=None,
    separate_rep_figures=True,
    skip_fit=False                # NEW parameter: if True, skip model fitting
):
    """
    Plot replicates for a given protein, plate, and treatment well with corresponding negative control.
    Also accumulates model metadata for later analysis.

    If skip_fit=True, the function bypasses the model fitting steps and plots only:
      - Raw fluorescence data,
      - Normalized fluorescence,
      - Derivative and normalized derivative curves.
    In this mode, fitted-curve annotations (e.g. Tₘ, dTₘ, model window) are omitted.
    """
    import matplotlib.pyplot as plt
    import numpy as np
    from scipy.signal import find_peaks
    import os

    if metadata_csv_filename is None:
        metadata_csv_filename = f"./output/metadata_{protein}_{plate}_{treatment_well}.csv"
    
    model_desc = {
        "s1": "Single Sigmoidal",
        "s1_d": "Single Sigmoidal w/Decay",
        "s2": "Double Sigmoidal",
        "s2_d": "Double Sigmoidal w/Decay",
        "mB": "Modified Boltz."
    }
    
    if candidate_models is None:
        candidate_models = list(MODEL_PARAMS.keys())
    
    reps = list(raw_data_dfs[protein][plate].keys())
    if verbose:
        print(f"Found replicates for {protein} {plate}: {reps}")
    
    first_ligand = None
    metadata_records = []
    neg_control_fits = {}  # caching negative control fits per replicate

    if separate_rep_figures:
        for i, rep in enumerate(reps):
            rep_data = raw_data_dfs[protein][plate][rep]
            if treatment_well not in rep_data or neg_control_well not in rep_data:
                if verbose:
                    print(f"{rep}: Missing required wells. Skipping.")
                continue

            df_treat = rep_data[treatment_well]
            df_neg = rep_data[neg_control_well]

            ligand_treat = get_ligand_for_plate_well(raw_data_dfs, plate, treatment_well)
            if first_ligand is None:
                first_ligand = ligand_treat
            ligand_neg = get_ligand_for_plate_well(raw_data_dfs, plate, neg_control_well)

            if not skip_fit:
                # Compute negative control best-fit for this replicate if not already cached.
                if rep in neg_control_fits:
                    best_fit_neg = neg_control_fits[rep]
                else:
                    best_fit_neg, _ = select_best_model_for_trace(
                        df_neg, candidate_models=candidate_models,
                        window_mode=window_mode, window_sizes=window_sizes,
                        show_plot=False, protein=protein, ligand=ligand_neg,
                        rep=rep, plate_ID=plate, well=neg_control_well,
                        fixed_window_size=fixed_window_size
                    )
                    neg_control_fits[rep] = best_fit_neg

                # Compute best-fit for treatment.
                best_fit_t, _ = select_best_model_for_trace(
                    df_treat, candidate_models=candidate_models,
                    window_mode=window_mode, window_sizes=window_sizes,
                    show_plot=False, protein=protein, ligand=ligand_treat,
                    rep=rep, plate_ID=plate, well=treatment_well,
                    fixed_window_size=fixed_window_size
                )
            else:
                best_fit_t = None
                best_fit_neg = None

            # Set up subplot layout as before.
            if include_tmchange_subplot and include_norm_subplot:
                nrows = 5
                height_ratios = [1, 3, 3, 1, 1]
                fig, axs = plt.subplots(nrows, 1, figsize=(10, 14), sharex=True,
                                        gridspec_kw={'height_ratios': height_ratios})
                ax_tmchange, ax_main, ax_norm, ax_deriv, ax_deriv_norm = axs
            elif include_tmchange_subplot and not include_norm_subplot:
                nrows = 3
                height_ratios = [1, 3, 1]
                fig, axs = plt.subplots(nrows, 1, figsize=(10, 10), sharex=True,
                                        gridspec_kw={'height_ratios': height_ratios})
                ax_tmchange, ax_main, ax_deriv = axs
            elif not include_tmchange_subplot and include_norm_subplot:
                nrows = 4
                height_ratios = [3, 3, 1, 1]
                fig, axs = plt.subplots(nrows, 1, figsize=(10, 12), sharex=True,
                                        gridspec_kw={'height_ratios': height_ratios})
                ax_main, ax_norm, ax_deriv, ax_deriv_norm = axs
            else:
                nrows = 2
                height_ratios = [3, 1]
                fig, axs = plt.subplots(nrows, 1, figsize=(10, 8), sharex=True,
                                        gridspec_kw={'height_ratios': height_ratios})
                ax_main, ax_deriv = axs

            rep_color = ["#0072B2", "#E69F00", "#009E73",
                         "#F0E442", "#56B4E9", "#D55E00", "#CC79A7"][i % 7]

            if include_tmchange_subplot:
                tmchange_ax = ax_tmchange
                tmchange_ax.set_ylabel("Comparison")
                tmchange_ax.set_ylim(-0.1, 1.1)

            # --- Plot Raw Data ---
            ax_main.plot(df_treat["Temperature"], df_treat["Fluorescence"],
                         color=rep_color, alpha=0.7, linestyle="-", linewidth=3,
                         label=f"{ligand_treat}")
            ax_main.plot(df_neg["Temperature"], df_neg["Fluorescence"],
                         color=rep_color, alpha=0.7, linestyle="--", linewidth=3,
                         label=f"{ligand_neg}")

            if not skip_fit and best_fit_t is not None:
                # Compute fitted curve for treatment (if fitting is enabled).
                t_start_fit = best_fit_t["t_start"]
                t_end_fit = best_fit_t["t_end"]
                df_treat_window = df_treat[(df_treat["Temperature"] >= t_start_fit) &
                                           (df_treat["Temperature"] <= t_end_fit)].copy()
                df_treat_window_norm = normalize_df(df_treat_window)
                t_min_fit, t_max_fit = df_treat_window_norm.attrs["T_bounds"]
                f_min_fit, f_max_fit = df_treat_window_norm.attrs["F_bounds"]
                popt_t = best_fit_t["popt"]
                model_func_t = MODEL_PARAMS[best_fit_t["model"]]["model_func"]
                temp_dense = np.linspace(t_min_fit, t_max_fit, 300)
                temp_dense_norm = (temp_dense - t_min_fit) / (t_max_fit - t_min_fit)
                y_fit_norm = model_func_t(temp_dense_norm, *popt_t)
                y_fit = y_fit_norm * (f_max_fit - f_min_fit) + f_min_fit

                if best_fit_neg is not None:
                    dTm_model = best_fit_t["Tm1"] - best_fit_neg["Tm1"]
                else:
                    dTm_model = np.nan

                label_model = (f"{ligand_treat} Fit (Tm={best_fit_t['Tm1']:.1f}°C, R²={best_fit_t['R2']:.3f}")
                if not np.isnan(dTm_model):
                    label_model += f", dTm={dTm_model:.1f}°C)"
                else:
                    label_model += ")"
                if not np.isnan(dTm_model) and dTm_model >= dTm_threshold:
                    label_model = r"$\mathbf{" + label_model + "}$"

                ax_main.plot(temp_dense, y_fit, color="black",
                             linestyle=":", label=label_model)
                t_norm_Tm = (best_fit_t["Tm1"] - t_min_fit) / (t_max_fit - t_min_fit)
                y_fit_Tm_raw = model_func_t(t_norm_Tm, *popt_t) * (f_max_fit - f_min_fit) + f_min_fit
                ax_main.plot(best_fit_t["Tm1"], y_fit_Tm_raw, 'o',
                             color=rep_color, markersize=8, markeredgecolor='black')
                ax_main.text(best_fit_t["Tm1"], y_fit_Tm_raw + 0.05*(f_max_fit - f_min_fit),
                             f"{best_fit_t['Tm1']:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')

            # --- Plot Normalized Data, Derivatives, etc. (always computed from raw data) ---
            if include_norm_subplot:
                df_treat_norm_all = normalize_df(df_treat)
                f_min_all, f_max_all = df_treat_norm_all.attrs["F_bounds"]
                ax_norm.plot(df_treat_norm_all["Temperature"], df_treat_norm_all["value_norm"],
                            color=rep_color, alpha=0.7, linestyle="-", linewidth=3,
                            label=f"{ligand_treat}")
                df_neg_norm_all = normalize_df(df_neg)
                ax_norm.plot(df_neg_norm_all["Temperature"], df_neg_norm_all["value_norm"],
                            color=rep_color, alpha=0.7, linestyle="--", linewidth=3,
                            label=f"{ligand_neg}")

            df_treat_norm_full = normalize_df(df_treat)
            df_treat_deriv = compute_derivative(df_treat_norm_full)
            norm_factor = (df_treat_norm_full.attrs["F_bounds"][1] - df_treat_norm_full.attrs["F_bounds"][0])
            df_treat_deriv["Derivative"] *= norm_factor

            df_neg_norm_full = normalize_df(df_neg)
            df_neg_deriv = compute_derivative(df_neg_norm_full)
            norm_factor_neg = (df_neg_norm_full.attrs["F_bounds"][1] - df_neg_norm_full.attrs["F_bounds"][0])
            df_neg_deriv["Derivative"] *= norm_factor_neg

            peaks_t = find_peaks(df_treat_deriv["Derivative"])[0]
            temp_global_t = np.nan
            if len(peaks_t) > 0:
                idx_tmax = np.argmax(df_treat_deriv["Derivative"].iloc[peaks_t])
                temp_global_t = df_treat_deriv["Temperature"].iloc[peaks_t[idx_tmax]]
            
            peaks_neg = find_peaks(df_neg_deriv["Derivative"])[0]
            temp_global_neg = np.nan
            if len(peaks_neg) > 0:
                idx_negmax = np.argmax(df_neg_deriv["Derivative"].iloc[peaks_neg])
                temp_global_neg = df_neg_deriv["Temperature"].iloc[peaks_neg[idx_negmax]]
            
            dTm_deriv = (temp_global_t - temp_global_neg) if (not np.isnan(temp_global_t) and not np.isnan(temp_global_neg)) else np.nan
            
            label_t_deriv = f"{ligand_treat}"
            if not np.isnan(temp_global_t):
                label_t_deriv += f" (Tm={temp_global_t:.1f}°C"
                if not np.isnan(dTm_deriv):
                    label_t_deriv += f", dTm={dTm_deriv:.1f}°C)"
                else:
                    label_t_deriv += ")"
            if not np.isnan(dTm_deriv) and dTm_deriv >= dTm_threshold:
                label_t_deriv = r"$\mathbf{" + label_t_deriv + "}$"
            
            ax_deriv.plot(df_treat_deriv["Temperature"], df_treat_deriv["Derivative"],
                          color=rep_color, linestyle="-", label=label_t_deriv)
            if not np.isnan(temp_global_t):
                d_val_t = df_treat_deriv.loc[df_treat_deriv["Temperature"] == temp_global_t, "Derivative"].values[0]
                ax_deriv.plot(temp_global_t, d_val_t, 'o', color=rep_color, markersize=8, markeredgecolor='black')
                ax_deriv.text(temp_global_t, d_val_t + 0.01*norm_factor,
                              f"{temp_global_t:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
            
            label_neg_deriv = f"{ligand_neg}"
            if not np.isnan(temp_global_neg):
                label_neg_deriv += f" (Tm={temp_global_neg:.1f}°C)"
            ax_deriv.plot(df_neg_deriv["Temperature"], df_neg_deriv["Derivative"],
                          color=rep_color, linestyle=":", label=label_neg_deriv)
            if not np.isnan(temp_global_neg):
                d_val_neg = df_neg_deriv.loc[df_neg_deriv["Temperature"] == temp_global_neg, "Derivative"].values[0]
                ax_deriv.plot(temp_global_neg, d_val_neg, 'o', color=rep_color, markersize=8, markeredgecolor='black')
                ax_deriv.text(temp_global_neg, d_val_neg + 0.01*norm_factor_neg,
                              f"{temp_global_neg:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
            
            cur_ylim = ax_deriv.get_ylim()
            ax_deriv.set_ylim(cur_ylim[0], cur_ylim[1] + 0.1*(cur_ylim[1]-cur_ylim[0]))
            ax_deriv.set_xlabel("Temperature (°C)")
            ax_deriv.set_ylabel("dF/dT (RFU/°C)")
            ax_deriv.legend(fontsize=8, loc="upper left")
            
            if include_norm_subplot:
                df_treat_deriv_norm = compute_derivative(df_treat_norm_full)
                label_t_deriv_norm = f"{ligand_treat}"
                if not np.isnan(temp_global_t):
                    label_t_deriv_norm += f" (Tm={temp_global_t:.1f}°C"
                    if not np.isnan(dTm_deriv):
                        label_t_deriv_norm += f", dTm={dTm_deriv:.1f}°C)"
                    else:
                        label_t_deriv_norm += ")"
                if not np.isnan(dTm_deriv) and dTm_deriv >= dTm_threshold:
                    label_t_deriv_norm = r"$\mathbf{" + label_t_deriv_norm + "}$"
                ax_deriv_norm.plot(df_treat_deriv_norm["Temperature"], df_treat_deriv_norm["Derivative"],
                                   color=rep_color, linestyle="-", label=label_t_deriv_norm)
                peaks_t_norm = find_peaks(df_treat_deriv_norm["Derivative"])[0]
                if len(peaks_t_norm) > 0:
                    t_pt_norm = df_treat_deriv_norm["Temperature"].iloc[peaks_t_norm[np.argmax(df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm])]]
                    d_pt_norm = df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm[np.argmax(df_treat_deriv_norm["Derivative"].iloc[peaks_t_norm])]]
                    ax_deriv_norm.plot(t_pt_norm, d_pt_norm, 'o', color=rep_color, markersize=8, markeredgecolor='black')
                    ax_deriv_norm.text(t_pt_norm, d_pt_norm + 0.01, f"{t_pt_norm:.1f}", fontsize=9, ha='center', color='black', fontweight='bold')
                
                df_neg_deriv_norm = compute_derivative(df_neg_norm_full)
                label_neg_deriv_norm = f"{ligand_neg}"
                ax_deriv_norm.plot(df_neg_deriv_norm["Temperature"], df_neg_deriv_norm["Derivative"],
                                   color=rep_color, linestyle=":", label=label_neg_deriv_norm)
                peaks_neg_norm = find_peaks(df_neg_deriv_norm["Derivative"])[0]
                if len(peaks_neg_norm) > 0:
                    t_pt_neg_norm = df_neg_deriv_norm["Temperature"].iloc[peaks_neg_norm[np.argmax(df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm])]]
                    d_pt_neg_norm = df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm[np.argmax(df_neg_deriv_norm["Derivative"].iloc[peaks_neg_norm])]]
                    ax_deriv_norm.plot(t_pt_neg_norm, d_pt_neg_norm, 'o', color=rep_color, markersize=8,
                                       markeredgecolor='black', alpha=0.5)
                    ax_deriv_norm.text(t_pt_neg_norm, d_pt_neg_norm + 0.01, f"{t_pt_neg_norm:.1f}",
                                       fontsize=9, ha='center', color='black', fontweight='bold')
                ax_deriv_norm.set_xlabel("Temperature (°C)")
            
            if include_tmchange_subplot:
                y_model = 0.3
                y_deriv = 0.7
                if (not skip_fit) and (best_fit_t is not None and best_fit_neg is not None):
                    exp_tm_model = best_fit_t["Tm1"]
                    ctrl_tm_model = best_fit_neg["Tm1"]
                    tmchange_ax.annotate("", xy=(exp_tm_model, y_model),
                                         xytext=(ctrl_tm_model, y_model),
                                         arrowprops=dict(arrowstyle="->", color="black"))
                    tmchange_ax.scatter(ctrl_tm_model, y_model, color='black')
                    tmchange_ax.scatter(exp_tm_model, y_model, color=rep_color, edgecolor='black')
                    delta_model = exp_tm_model - ctrl_tm_model
                    label_model_change = f"Model ΔTm={delta_model:.1f}"
                    if delta_model >= dTm_threshold:
                        label_model_change = r"$\mathbf{" + label_model_change + "}$"
                    tmchange_ax.text(max(exp_tm_model, ctrl_tm_model)+0.5, y_model,
                                     label_model_change, va='center', fontsize=8)
                
                if (not skip_fit) and (not np.isnan(temp_global_t) and not np.isnan(temp_global_neg)):
                    tmchange_ax.annotate("", xy=(temp_global_t, y_deriv),
                                         xytext=(temp_global_neg, y_deriv),
                                         arrowprops=dict(arrowstyle="->", color="black"))
                    tmchange_ax.scatter(temp_global_neg, y_deriv, color='black')
                    tmchange_ax.scatter(temp_global_t, y_deriv, color=rep_color, edgecolor='black')
                    delta_deriv = temp_global_t - temp_global_neg
                    label_deriv_change = f"Deriv ΔTm={delta_deriv:.1f}"
                    if delta_deriv >= dTm_threshold:
                        label_deriv_change = r"$\mathbf{" + label_deriv_change + "}$"
                    tmchange_ax.text(max(temp_global_t, temp_global_neg)+0.5, y_deriv,
                                     label_deriv_change, va='center', fontsize=8)
                
                tmchange_ax.set_yticks([y_model, y_deriv])
                tmchange_ax.set_yticklabels(["Model", "Derivative"])
            
            if not skip_fit:
                if best_fit_t is not None:
                    treat_model_key = best_fit_t["model"]
                    treat_model_name = model_desc.get(treat_model_key, treat_model_key)
                else:
                    treat_model_name = "N/A"
                if best_fit_neg is not None:
                    ctrl_model_key = best_fit_neg["model"]
                    ctrl_model_name = model_desc.get(ctrl_model_key, ctrl_model_key)
                else:
                    ctrl_model_name = "N/A"
                sub_title = f"Models: {treat_model_name} (treatment), {ctrl_model_name} (control)"
            else:
                sub_title = "Data only (no fit)"
            
            ax_main.set_xlabel("")
            ax_main.set_ylabel("Fluorescence (RFU)")
            ax_main.legend(fontsize=8, loc="upper left")
            
            if include_norm_subplot:
                ax_norm.set_xlabel("")
                ax_norm.set_ylabel("Normalized Fluorescence")
                ax_norm.legend(fontsize=8, loc="upper left")
                ax_deriv.set_xlabel("")
                ax_deriv.set_ylabel("dF/dT (RFU/°C)")
                ax_deriv.legend(fontsize=8, loc="upper left")
                ax_deriv_norm.set_ylabel("Normalized dF/dT")
                ax_deriv_norm.legend(fontsize=8, loc="upper left")
            else:
                ax_deriv.set_xlabel("Temperature (°C)")
                ax_deriv.set_ylabel("dF/dT (RFU/°C)")
                ax_deriv.legend(fontsize=8, loc="upper left")
            
            fig.suptitle(
                f"{protein}, {ligand_treat} ({plate}, {treatment_well}) - Replicate {rep}\n{sub_title}",
                fontsize=14, y=0.98
            )
            
            plt.tight_layout(rect=[0, 0, 1, 0.93])
            
            if save_png:
                if png_filename is None:
                    base_filename = f"{protein}_{plate}_{treatment_well}"
                else:
                    base_filename = os.path.splitext(png_filename)[0]
                rep_filename = f"./output_pngs/{base_filename}_rep{rep}.png"
                plt.savefig(rep_filename, dpi=300)
                if verbose:
                    print(f"Figure saved as {rep_filename}")
            
            # Save metadata for this replicate.
            record = {
                "protein": protein,
                "plate": plate,
                "replicate": rep,
                "treatment_well": treatment_well,
                "ligand_treat": ligand_treat,
                "neg_control_well": neg_control_well,
                "ligand_neg": ligand_neg,
                "Tm_treatment_model": best_fit_t["Tm1"] if (not skip_fit and best_fit_t is not None) else np.nan,
                "Tm_control_model": best_fit_neg["Tm1"] if (not skip_fit and best_fit_neg is not None) else np.nan,
                "dTm_model": (best_fit_t["Tm1"] - best_fit_neg["Tm1"]) if (not skip_fit and best_fit_t is not None and best_fit_neg is not None) else np.nan,
                "Tm_treatment_deriv": temp_global_t if not np.isnan(temp_global_t) else np.nan,
                "Tm_control_deriv": temp_global_neg if not np.isnan(temp_global_neg) else np.nan,
                "dTm_deriv": dTm_deriv,
                "model_window": (t_start_fit, t_end_fit) if (not skip_fit and best_fit_t is not None) else None,
                "model_used": best_fit_t["model"] if (not skip_fit and best_fit_t is not None) else "N/A",
                "R2": best_fit_t["R2"] if (not skip_fit and best_fit_t is not None) else np.nan
            }
            metadata_records.append(record)
            
            plt.close(fig)
        
        metadata_df = pd.DataFrame(metadata_records)
        if save_metadata_csv:
            final_metadata_fn=f"{metadata_csv_filename}_final"
            metadata_df.to_csv(final_metadata_fn, index=False)
            if verbose:
                print(f"Metadata saved as {final_metadata_fn}")
        return metadata_df

In [ ]:
import os
import pandas as pd

output_dir = "./output"
master_metadata_filename = os.path.join(output_dir, "master_metadata.csv")
final_metadata_filename = os.path.join(output_dir, "master_metadata_final.csv")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created directory: {output_dir}")

if os.path.exists(master_metadata_filename):
    master_existing = pd.read_csv(master_metadata_filename)
    processed_keys = set(master_existing[['protein', 'plate', 'treatment_well']].apply(tuple, axis=1))
    print(f"Found existing master metadata with {len(processed_keys)} entries.")
else:
    processed_keys = set()

master_metadata_list = []

for protein in raw_data_dfs:
    for plate in raw_data_dfs[protein]:
        protein_plate_metadata_list = []
        # Check if the plate dictionary is empty.
        reps_dict = raw_data_dfs[protein][plate]
        if not reps_dict:
            print(f"No replicates found for Protein: {protein}, Plate: {plate}. Skipping.")
            continue
        
        first_rep = next(iter(reps_dict.keys()))
        wells = raw_data_dfs[protein][plate][first_rep].keys()
        
        for treatment_well in wells:
            if treatment_well == "A01":
                continue
            key = (protein, plate, treatment_well)
            if key in processed_keys:
                print(f"Skipping already processed: Protein {protein}, Plate {plate}, Well {treatment_well}")
                continue
            
            print(f"Processing Protein: {protein}, Plate: {plate}, Well: {treatment_well}")
            try:
                metadata_df = plot_reps_and_neg_control_sep(
                    protein=protein,
                    plate=plate,
                    treatment_well=treatment_well,
                    neg_control_well="A01",
                    raw_data_dfs=raw_data_dfs,
                    candidate_models=["mB"],
                    window_mode="full",
                    window_sizes=[50, 60, 70],
                    verbose=False,
                    save_png=True,
                    include_norm_subplot=True,
                    save_metadata_csv=True,
                    metadata_csv_filename=None,
                    separate_rep_figures=True, 
                    skip_fit=True, 
                    include_tmchange_subplot=False
                )
                protein_plate_metadata_list.append(metadata_df)
                master_metadata_list.append(metadata_df)
                processed_keys.add(key)
                
                if os.path.exists(master_metadata_filename):
                    metadata_df.to_csv(master_metadata_filename, mode='a', header=False, index=False)
                else:
                    metadata_df.to_csv(master_metadata_filename, mode='w', header=True, index=False)
                print(f"Updated master metadata file: {master_metadata_filename}")
            except Exception as e:
                print(f"Error processing {protein}, {plate}, {treatment_well}: {e}")
                continue
        
        if protein_plate_metadata_list:
            pp_metadata = pd.concat(protein_plate_metadata_list, ignore_index=True)
            pp_filename = os.path.join(output_dir, f"metadata_{protein}_{plate}.csv")
            pp_metadata.to_csv(pp_filename, index=False)
            print(f"Saved metadata for {protein} {plate} as {pp_filename}")

if master_metadata_list:
    master_metadata = pd.concat(master_metadata_list, ignore_index=True)
    master_metadata.to_csv(final_metadata_filename, index=False)
    print(f"Master metadata saved as {final_metadata_filename}")

# THINGS TO DO

[] Increase speed of analysis (e.g. remove unncessary negative control reanalysis)
[] dictionary protein well to UniProt?
[] SMILES and chemical structure viewing
[] Documentation
[] Clean up code
[] Fix output folders and naming to prevent overriding
[] plotly export fix scrunched up x-axis
[] cells to export certain traces
[] plotly interactive graph with overlay of trace on hover?
[] plotly data to 2 sig figs
[] option to reanalyse specific conditions and override master dataframe csv
[] webserver
[] paper/technical note
[] github upload with example data

# Plot dTm Values for each plate

In [30]:
import plotly.express as px
import pandas as pd

def plot_dTm_scatter(metadatafile, protein, plate_id, dtm_option="M", vertical=False, hit_threshold=2.0):
    """
    Creates an interactive scatter plot of dTm values for a specified protein and plate.
    
    Parameters:
      metadatafile (str): Path to the master metadata CSV file.
      protein (str): Protein identifier to filter by.
      plate_id (str): Plate identifier to filter by.
      dtm_option (str): "M" for model-based dTm, "D" for derivative-based dTm.
      vertical (bool): If True, plots vertically (x-axis=dTm, y-axis=Well_Ligand). 
                       If False, plots horizontally (x-axis=Well_Ligand, y-axis=dTm).
      hit_threshold (float): dTm threshold value; a dotted line is drawn at this value.
    
    Returns:
      fig (plotly.graph_objs._figure.Figure): The Plotly figure object.
    """
    
    # Read the master metadata CSV
    df_master = pd.read_csv(metadatafile)
    
    # Rename columns to our plotting names.
    df_master = df_master.rename(columns={
        "ligand_treat": "Ligand",
        "treatment_well": "Well",
        "dTm_model": "dTm M",      # model-based dTm
        "Tm_treatment_model": "Tm M",  # model-based Tm
        "dTm_deriv": "dTm D",      # derivative-based dTm
        "Tm_treatment_deriv": "Tm D",  # derivative-based Tm
        "replicate": "Replicate"
    })
    
    # Filter for the desired protein and plate.
    df_filtered = df_master[(df_master["protein"] == protein) & (df_master["plate"] == plate_id)]
    
    # Choose the column for the chosen dTm metric.
    y_col = f"dTm {dtm_option}"
    df_filtered.dropna(subset=[y_col], inplace=True)
    
    if not vertical:
        # Create a new column "Well_Ligand" that reports the well position (in parentheses)
        # before the ligand name.
        df_filtered["Well_Ligand"] = df_filtered["Well"].astype(str) + "   " + df_filtered["Ligand"].astype(str)
        x_col = "Well_Ligand"
    else:
        # For vertical orientation, you might decide whether to use the same.
        # Here, we continue using "Ligand" only.
        x_col = "Ligand"
    
    # Determine unique x-axis categories and create custom tick labels.
    unique_categories = list(df_filtered[x_col].unique())
    tick_text = []
    for cat in unique_categories:
        subset = df_filtered[df_filtered[x_col] == cat]
        if (subset[y_col] > hit_threshold).all():
            tick_text.append(f"<b>{cat}</b>")
        else:
            tick_text.append(cat)
    
    # Create the scatter plot based on orientation.
    if vertical:
        # Vertical orientation: x-axis is dTm metric, y-axis is the chosen category.
        fig = px.scatter(
            df_filtered,
            x=y_col,
            y=x_col,
            color="Replicate",
            hover_data=["Well", "Tm M", "Tm D"],
            title=f"{protein} {plate_id} - All Replicates (dTm {dtm_option})"
        )
        fig.update_xaxes(title_text=f"dTm {dtm_option} (°C)")
        fig.update_yaxes(
            tickmode="array", 
            tickvals=unique_categories, 
            ticktext=tick_text,
            tickangle=45,
            tickfont=dict(size=10)
        )
        # Add a vertical dotted line for the hit threshold.
        fig.add_shape(
            type="line",
            x0=hit_threshold, x1=hit_threshold,
            y0=0, y1=1,
            xref="x", yref="paper",
            line=dict(color="black", width=2, dash="dot")
        )
    else:
        # Horizontal orientation: x-axis is the chosen category, y-axis is the dTm metric.
        fig = px.scatter(
            df_filtered,
            x=x_col,
            y=y_col,
            color="Replicate",
            hover_data=["Well", "Tm M", "Tm D"],
            title=f"{protein} {plate_id} - All Replicates (dTm {dtm_option})"
        )
        fig.update_yaxes(title_text=f"dTm {dtm_option} (°C)")
        fig.update_xaxes(
            tickmode="array", 
            tickvals=unique_categories, 
            ticktext=tick_text,
            tickangle=90,
            tickfont=dict(size=10)
        )
        # Add a horizontal dotted line for the hit threshold.
        fig.add_shape(
            type="line",
            x0=0, x1=1,
            y0=hit_threshold, y1=hit_threshold,
            xref="paper", yref="y",
            line=dict(color="black", width=2, dash="dot")
        )
    
    return fig

In [ ]:

# Example usage:
fig = plot_dTm_scatter(
    metadatafile="./output/master_metadata.csv",
    protein="B5",
    plate_id="PM5",
    dtm_option="D",  # or "D" for derivative-based
    vertical=False,
    hit_threshold=2.0
)
fig.show()

In [ ]:
import os
import pandas as pd

# Create the directory for saving tm scatter PNGs if it doesn't exist.
tm_scatter_dir = "./output/tm_scatter_pngs"
if not os.path.exists(tm_scatter_dir):
    os.makedirs(tm_scatter_dir)
    print(f"Created directory: {tm_scatter_dir}")

# Define the master metadata file (assumed to have been created previously).
master_metadata_file = "./output/master_metadata.csv"

# Check if the master metadata file exists.
if not os.path.exists(master_metadata_file):
    print("Master metadata file does not exist. Exiting.")
    exit(1)

# Read the master metadata file once.
df_master = pd.read_csv(master_metadata_file)

# Loop over each protein and plate in raw_data_dfs.
# (raw_data_dfs is assumed to be your nested dictionary containing the raw data.)
for protein in raw_data_dfs:
    for plate in raw_data_dfs[protein]:
        # Check if there is any data for the current protein and plate.
        df_subset = df_master[(df_master["protein"] == protein) & (df_master["plate"] == plate)]
        if df_subset.empty:
            print(f"No data found for {protein} {plate}. Skipping plotting.")
            continue
        
        try:
            # Generate the Plotly scatter plot using the master metadata file.
            fig = plot_dTm_scatter(
                metadatafile=master_metadata_file,
                protein=protein,
                plate_id=plate,
                dtm_option="D",   # Change to "D" for derivative-based dTm if desired.
                vertical=False,   # Set True to flip axes.
                hit_threshold=2.0
            )
            fig.show()
            # Build the PNG filename.
            png_filename = os.path.join(tm_scatter_dir, f"{protein}_{plate}_tm_scatter.png")
            fig.update_layout(
                width=1800, 
                height=600,
                margin=dict(b=100)  # Increase bottom margin to ensure labels are not cut off
            )
            # Save the figure as a PNG image.
            fig.write_image(png_filename, scale=2)
            print(f"Saved tm scatter figure for {protein} {plate} to {png_filename}")
        except Exception as e:
            print(f"Error processing {protein} {plate}: {e}")

conda install -c conda-forge python-kaleido

In [ ]:
!pip install -U kaleido